# Stage 3 XGBoost IDS Training

Stage 3 trains a supervised machine-learning classifier for IDS flow classification using CSE-CIC-IDS2018 flow records.

This stage is separate from Stage 2 signature detection. Stage 2 produces interpretable rule evidence, while Stage 3 produces an ML detection signal. The Stage 3 output will later be fused with Stage 2 signature evidence in the Stage 4 Fusion Engine.

Feature-label separation must be preserved:

- Training can use labels.
- Prediction must use feature-only inputs.
- Evaluation can join ground truth only after prediction.

## Environment Setup

This section installs and imports the libraries needed in Google Colab. Raw CSE-CIC-IDS2018 files are large, so keep dataset archives outside GitHub.

In [ ]:
# Colab-friendly dependency install. In local environments, install these in a virtual environment instead.
%pip install -q xgboost kagglehub


In [ ]:
import json
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Dataset Input Options

Choose one dataset source by setting `DATASET_SOURCE` in the next cell.

Option A: upload `archive.zip` directly to Colab and use:

```txt
/content/archive.zip
```

Option B: store `archive.zip` in Google Drive, mount Drive, and use:

```txt
/content/drive/MyDrive/cse-cic-ids2018/archive.zip
```

Option C: download the Kaggle dataset with KaggleHub:

```python
kagglehub.dataset_download('solarmainframe/ids-intrusion-csv')
```

Option D: use an already extracted folder.

Do not hardcode a private Windows path in the notebook. Raw CSV files and dataset archives should not be committed to GitHub.


In [ ]:
# Choose one: 'kagglehub', 'upload_zip', 'drive_zip', or 'extracted_folder'.
DATASET_SOURCE = 'kagglehub'

# Option A: uploaded archive in Colab.
UPLOAD_ARCHIVE_PATH = Path('/content/archive.zip')

# Option B: Google Drive archive. Uncomment after mounting Drive.
# from google.colab import drive
# drive.mount('/content/drive')
DRIVE_ARCHIVE_PATH = Path('/content/drive/MyDrive/cse-cic-ids2018/archive.zip')

# Option C: KaggleHub dataset identifier.
KAGGLEHUB_DATASET = 'solarmainframe/ids-intrusion-csv'

# Option D: already extracted folder.
EXTRACTED_DATA_DIR = Path('/content/cse-cic-ids2018')

# Keep this cap while testing in Colab. Set to None only if memory is sufficient.
ROW_CAP_PER_CSV = 200_000


## Load CSE-CIC-IDS2018 CSV Files

This section opens the archive if needed, finds CSV files recursively, loads them into pandas, and prints loaded file names and row counts.

Full CSE-CIC-IDS2018 is large. Use `ROW_CAP_PER_CSV` to avoid Colab memory issues during development.

In [ ]:
def resolve_dataset_dir() -> Path:
    if DATASET_SOURCE == 'kagglehub':
        dataset_path = Path(kagglehub.dataset_download(KAGGLEHUB_DATASET))
        print(f'KaggleHub dataset path: {dataset_path}')
        return dataset_path

    if DATASET_SOURCE == 'upload_zip':
        archive_path = UPLOAD_ARCHIVE_PATH
    elif DATASET_SOURCE == 'drive_zip':
        archive_path = DRIVE_ARCHIVE_PATH
    elif DATASET_SOURCE == 'extracted_folder':
        if not EXTRACTED_DATA_DIR.exists():
            raise FileNotFoundError(f'Extracted dataset folder not found: {EXTRACTED_DATA_DIR}')
        return EXTRACTED_DATA_DIR
    else:
        raise ValueError(f'Unsupported DATASET_SOURCE: {DATASET_SOURCE}')

    if not archive_path.exists():
        raise FileNotFoundError(f'Dataset archive not found: {archive_path}')

    EXTRACTED_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, 'r') as archive:
        archive.extractall(EXTRACTED_DATA_DIR)
    return EXTRACTED_DATA_DIR


def find_csv_files(dataset_dir: Path) -> list[Path]:
    csv_files = sorted(dataset_dir.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found under {dataset_dir}')
    return csv_files


def load_csv_files(csv_files: list[Path], row_cap_per_csv: int | None = ROW_CAP_PER_CSV) -> pd.DataFrame:
    frames = []
    loaded_rows = 0
    for csv_path in csv_files:
        print(f'Loading {csv_path.name}')
        frame = pd.read_csv(csv_path, nrows=row_cap_per_csv, low_memory=False)
        frame['sourceFile'] = csv_path.name
        frames.append(frame)
        loaded_rows += len(frame)
        print(f'  rows loaded: {len(frame):,}')

    combined = pd.concat(frames, ignore_index=True)
    print(f'Total loaded rows: {loaded_rows:,}')
    return combined

DATASET_DIR = resolve_dataset_dir()
csv_files = find_csv_files(DATASET_DIR)
print('CSV files found:')
for csv_file in csv_files:
    print('-', csv_file.name)

raw_df = load_csv_files(csv_files)
raw_df.shape


## Clean and Validate Data

Cleaning steps:

- Strip column names.
- Replace `inf` and `-inf` with `NaN`.
- Drop rows with missing labels.
- Handle mixed CSV schemas.
- Keep only rows with known mapped labels.
- Remove rows with invalid numeric feature values.

In [ ]:
LABEL_COLUMN = 'Label'

FORBIDDEN_PREDICTION_FIELDS = {
    'Label',
    'rawLabel',
    'attackType',
    'mappedAttackType',
    'groundTruth',
    'severity',
    'similarityKey',
}

NON_FEATURE_COLUMNS = {
    'id',
    'timestamp',
    'Timestamp',
    'sourceFile',
    'Flow ID',
    'Src IP',
    'SrcIP',
    'Source IP',
    'sourceIp',
    'Dst IP',
    'DstIP',
    'Destination IP',
    'destinationIp',
}

initial_rows = len(raw_df)
clean_df = raw_df.copy()
clean_df.columns = clean_df.columns.str.strip()

if LABEL_COLUMN not in clean_df.columns:
    raise ValueError(f'Missing required label column: {LABEL_COLUMN}')

clean_df = clean_df.replace([np.inf, -np.inf], np.nan)
missing_label_rows = clean_df[LABEL_COLUMN].isna().sum()
clean_df = clean_df.dropna(subset=[LABEL_COLUMN])
clean_df[LABEL_COLUMN] = clean_df[LABEL_COLUMN].astype(str).str.strip()

print(f'Initial rows: {initial_rows:,}')
print(f'Removed rows with missing labels: {missing_label_rows:,}')
print(f'Rows after label cleanup: {len(clean_df):,}')

## Map Raw Labels to Dashboard Attack Types

The mapping is kept consistent with Stage 1. Spelling variants such as `Infilteration` are handled explicitly.

In [ ]:
LABEL_MAPPING = {
    'Benign': 'Benign',
    'FTP-BruteForce': 'Brute Force',
    'SSH-Bruteforce': 'Brute Force',
    'Bot': 'Botnet',
    'DoS attacks-GoldenEye': 'DoS',
    'DoS attacks-Slowloris': 'DoS',
    'DoS attacks-SlowHTTPTest': 'DoS',
    'DoS attacks-Hulk': 'DoS',
    'DDoS attacks-LOIC-HTTP': 'DDoS',
    'DDOS attack-HOIC': 'DDoS',
    'DDOS attack-LOIC-UDP': 'DDoS',
    'Brute Force -Web': 'Web Attack',
    'Brute Force -XSS': 'Web Attack',
    'SQL Injection': 'Web Attack',
    'Infilteration': 'Infiltration',
    'Infiltration': 'Infiltration',
}

print('Raw label distribution:')
print(clean_df[LABEL_COLUMN].value_counts())

clean_df['mappedAttackType'] = clean_df[LABEL_COLUMN].map(LABEL_MAPPING)
unmapped_labels = sorted(clean_df.loc[clean_df['mappedAttackType'].isna(), LABEL_COLUMN].unique())
print('Unknown / unmapped labels:', unmapped_labels if unmapped_labels else 'none')

before_known_filter = len(clean_df)
clean_df = clean_df.dropna(subset=['mappedAttackType']).copy()
print(f'Removed rows with unmapped labels: {before_known_filter - len(clean_df):,}')

print('Mapped attack type distribution:')
print(clean_df['mappedAttackType'].value_counts())

## Select Feature Columns

Only observable numeric flow features are selected.

Excluded fields include labels, raw/mapped attack type, timestamp, source/destination IPs, and dashboard-derived fields such as severity, risk score, ground truth, and similarity key.

The final selected feature names are saved to `feature-columns.json` during artifact export.

In [ ]:
def select_numeric_feature_columns(dataframe: pd.DataFrame) -> list[str]:
    excluded = FORBIDDEN_PREDICTION_FIELDS | NON_FEATURE_COLUMNS | {'mappedAttackType'}
    candidates = []
    for column in dataframe.columns:
        if column in excluded:
            continue

        numeric_series = pd.to_numeric(dataframe[column], errors='coerce')
        numeric_series = numeric_series.replace([np.inf, -np.inf], np.nan)

        # Keep columns that are mostly numeric after coercion and infinity cleanup.
        if numeric_series.notna().mean() >= 0.95:
            dataframe[column] = numeric_series
            candidates.append(column)
    return candidates


def remove_invalid_feature_rows(dataframe: pd.DataFrame, selected_features: list[str]) -> pd.DataFrame:
    cleaned = dataframe.copy()
    cleaned[selected_features] = cleaned[selected_features].apply(pd.to_numeric, errors='coerce')
    cleaned[selected_features] = cleaned[selected_features].replace([np.inf, -np.inf], np.nan)

    # XGBoost cannot handle infinity or extremely large float values.
    max_float32 = np.finfo(np.float32).max
    too_large_mask = cleaned[selected_features].abs().gt(max_float32).any(axis=1)
    if too_large_mask.any():
        print(f'Removed rows with values too large for XGBoost float32 handling: {int(too_large_mask.sum()):,}')
        cleaned = cleaned.loc[~too_large_mask].copy()

    before_drop = len(cleaned)
    cleaned = cleaned.dropna(subset=selected_features + ['mappedAttackType']).copy()
    print(f'Removed rows with NaN / invalid numeric feature values: {before_drop - len(cleaned):,}')
    return cleaned


feature_columns = select_numeric_feature_columns(clean_df)
if not feature_columns:
    raise ValueError('No numeric feature columns selected. Check CSV schema and cleaning rules.')

before_numeric_drop = len(clean_df)
model_df = remove_invalid_feature_rows(clean_df, feature_columns)
print(f'Total rows removed during numeric feature cleanup: {before_numeric_drop - len(model_df):,}')
print(f'Selected feature count: {len(feature_columns)}')
print(feature_columns)

# Final safety check before sampling/training.
if not np.isfinite(model_df[feature_columns].to_numpy(dtype=np.float64)).all():
    raise ValueError('Feature matrix still contains inf, -inf, NaN, or too-large values after cleanup.')


## Sampling Strategy

The full dataset is large and imbalanced. This section supports controlled class sampling so Colab training stays manageable.

Monitor class imbalance before trusting model results. Sampling is for development and must be described in the report.

In [ ]:
MAX_ROWS_PER_CLASS = 20_000

sampled_df = (
    model_df
    .groupby('mappedAttackType', group_keys=False)
    .apply(lambda group: group.sample(n=min(len(group), MAX_ROWS_PER_CLASS), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print('Sampled class distribution:')
print(sampled_df['mappedAttackType'].value_counts())

## Train / Test Split

Use a stratified split where possible so each attack type is represented in both training and test sets.

In [ ]:
X = sampled_df[feature_columns].copy()
X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)

finite_row_mask = np.isfinite(X.to_numpy(dtype=np.float64)).all(axis=1)
if not finite_row_mask.all():
    removed_rows = int((~finite_row_mask).sum())
    print(f'Removed sampled rows with non-finite feature values before split: {removed_rows:,}')
    X = X.loc[finite_row_mask].copy()
    sampled_df = sampled_df.loc[finite_row_mask].copy()

y = sampled_df['mappedAttackType']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

stratify_target = y_encoded if pd.Series(y_encoded).value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y_encoded,
    sampled_df.index.astype(str),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Classes:', list(label_encoder.classes_))


## Train XGBoost Classifier

These are starter parameters for Stage 3 experimentation. They are not claimed to be final or production-optimal.

In [ ]:
if not np.isfinite(X_train.to_numpy(dtype=np.float64)).all():
    raise ValueError('X_train contains inf, -inf, NaN, or too-large values. Re-run cleaning cells before training.')
if not np.isfinite(X_test.to_numpy(dtype=np.float64)).all():
    raise ValueError('X_test contains inf, -inf, NaN, or too-large values. Re-run cleaning cells before training.')

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train)
print('XGBoost model trained.')


## Evaluate Model

Ground truth labels are used here only after prediction for evaluation.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
report_dict = classification_report(y_test, y_pred, target_names=label_encoder.classes_, output_dict=True, zero_division=0)
report_text = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred).tolist()
precision, recall, f1, support = precision_recall_fscore_support(
    y_test,
    y_pred,
    labels=list(range(len(label_encoder.classes_))),
    zero_division=0,
)

print('Accuracy:', accuracy)
print(report_text)
print('Confusion matrix:')
print(conf_matrix)

false_positive_notes = []
false_negative_notes = []
for class_index, class_name in enumerate(label_encoder.classes_):
    fp = int(((y_pred == class_index) & (y_test != class_index)).sum())
    fn = int(((y_pred != class_index) & (y_test == class_index)).sum())
    false_positive_notes.append({'class': class_name, 'falsePositiveCount': fp})
    false_negative_notes.append({'class': class_name, 'falseNegativeCount': fn})

print('False positive notes:', false_positive_notes)
print('False negative notes:', false_negative_notes)

evaluation_summary = {
    'accuracy': float(accuracy),
    'classificationReport': report_dict,
    'confusionMatrix': conf_matrix,
    'perClass': [
        {
            'class': class_name,
            'precision': float(precision[index]),
            'recall': float(recall[index]),
            'f1': float(f1[index]),
            'support': int(support[index]),
        }
        for index, class_name in enumerate(label_encoder.classes_)
    ],
    'falsePositiveNotes': false_positive_notes,
    'falseNegativeNotes': false_negative_notes,
}

## Export Model Artifacts

Artifacts are exported to a local Colab folder first:

```txt
/content/stage-3-artifacts/models/
```

Only model artifacts and small JSON outputs should be copied back to the repository. Raw CSV files and `archive.zip` must not be committed.

In [ ]:
ARTIFACT_ROOT = Path('/content/stage-3-artifacts')
MODEL_DIR = ARTIFACT_ROOT / 'models'
OUTPUT_DIR = ARTIFACT_ROOT / 'outputs'
EVALUATION_DIR = ARTIFACT_ROOT / 'evaluation'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

model.save_model(MODEL_DIR / 'xgboost_ids_model.json')

feature_columns_path = MODEL_DIR / 'feature-columns.json'
feature_columns_path.write_text(json.dumps(feature_columns, indent=2), encoding='utf-8')

label_mapping_artifact = {str(index): class_name for index, class_name in enumerate(label_encoder.classes_)}
(MODEL_DIR / 'label-mapping.json').write_text(json.dumps(label_mapping_artifact, indent=2), encoding='utf-8')

preprocessing_config = {
    'selectedFeatures': feature_columns,
    'forbiddenFields': sorted(FORBIDDEN_PREDICTION_FIELDS),
    'randomState': RANDOM_STATE,
    'samplingSettings': {'maxRowsPerClass': MAX_ROWS_PER_CLASS},
    'labelMapping': LABEL_MAPPING,
    'notes': [
        'Training uses labels for supervised learning.',
        'Prediction must use feature-only records.',
        'Ground truth is joined only after prediction for evaluation.',
    ],
}
(MODEL_DIR / 'preprocessing-config.json').write_text(json.dumps(preprocessing_config, indent=2), encoding='utf-8')

(EVALUATION_DIR / 'ml-evaluation-summary.json').write_text(json.dumps(evaluation_summary, indent=2), encoding='utf-8')

summary_md = [
    '# Stage 3 ML Evaluation Summary',
    '',
    f'- Accuracy: {accuracy:.4f}',
    f'- Test records: {len(y_test)}',
    '',
    '## Classification Report',
    '',
    '```txt',
    report_text,
    '```',
    '',
    '## False Positive Notes',
]
for item in false_positive_notes:
    summary_md.append(f"- {item['class']}: {item['falsePositiveCount']}")
summary_md.extend(['', '## False Negative Notes'])
for item in false_negative_notes:
    summary_md.append(f"- {item['class']}: {item['falseNegativeCount']}")

summary_text = "\n".join(summary_md) + "\n"
(EVALUATION_DIR / 'ml-evaluation-summary.md').write_text(summary_text, encoding='utf-8')

print('Artifacts exported to:', ARTIFACT_ROOT)


## Generate Sample ML Predictions

Prediction uses feature-only records. The held-out `X_test` features are used for inference first, then labels may be referenced separately for evaluation.

Prediction objects must not include ground truth.

In [ ]:
test_probabilities = model.predict_proba(X_test)
test_predictions = model.predict(X_test)

prediction_records = []
for row_number, (row_id, predicted_class_index, probabilities) in enumerate(zip(id_test, test_predictions, test_probabilities), start=1):
    confidence = float(np.max(probabilities))
    prediction_records.append({
        'id': str(row_id) if row_id is not None else f'ML-{row_number:04d}',
        'predictedAttackType': str(label_encoder.inverse_transform([predicted_class_index])[0]),
        'modelConfidence': round(confidence, 6),
        'baseRiskScore': int(round(confidence * 100)),
    })

prediction_path = OUTPUT_DIR / 'ml-predictions.sample.json'
prediction_path.write_text(json.dumps(prediction_records, indent=2), encoding='utf-8')
print(f'Wrote {len(prediction_records)} predictions to {prediction_path}')
prediction_records[:3]

## How to Copy Artifacts Back to Repository

After Colab training, download or copy these small artifacts back into the repository:

```txt
stage-3/models/xgboost_ids_model.json
stage-3/models/feature-columns.json
stage-3/models/label-mapping.json
stage-3/models/preprocessing-config.json
stage-3/outputs/ml-predictions.sample.json
stage-3/evaluation/ml-evaluation-summary.json
stage-3/evaluation/ml-evaluation-summary.md
```

Do not copy raw CSE-CIC-IDS2018 CSV files or `archive.zip` into Git.

## Notes and Limitations

- Training on sampled data may not represent full production traffic.
- CSE-CIC-IDS2018 is labelled benchmark data, not live deployment traffic.
- The model should be evaluated on held-out data.
- Real deployment would require consistent flow feature extraction.
- Stage 3 does not replace Stage 2; both will be fused later.
- Human feedback adaptation is still a later stage.
- This notebook does not claim production-ready IDS performance.